# PURITY Inference

Runs the unified config-driven inference pipeline for PURITY.

In [1]:
from __future__ import annotations

from pathlib import Path

import pyarrow.parquet as pq

from pioneerml.integration.zenml import load_step_output
from pioneerml.integration.zenml import utils as zenml_utils

PROJECT_ROOT = zenml_utils.setup_repo_pythonpath(Path(zenml_utils.find_project_root()).resolve())
from pioneerml_purity_plugin.purity.pipeline import inference_pipeline, load_config

zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)


Using ZenML repository root: /workspace
Ensure this is the top-level of your repo (.zen must live here).


## Build Input + Resolve Model

In [2]:
DATA_DIR = PROJECT_ROOT / 'data' / 'purity_inputs'
USE_ALL_FILES = True
SINGLE_FILE = 'all_ml_000.parquet'
MAX_FILES = None  # Set an int only when USE_ALL_FILES=True.

# Inference-handle toggle: TorchScript is the default path.
USE_TORCHSCRIPT_HANDLE = True


def resolve_input_sources(data_dir: Path, *, use_all_files: bool, single_file: str, max_files: int | None = None) -> list[str]:
    if use_all_files:
        paths = sorted(data_dir.glob('*.parquet'))
        if max_files is not None:
            paths = paths[: int(max_files)]
    else:
        paths = [data_dir / single_file]

    missing = [str(p) for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError('Missing parquet input file(s):\n' + '\n'.join(missing))

    return [str(p.resolve()) for p in paths]


def resolve_model_artifact(project_root: Path, *, use_torchscript: bool) -> tuple[Path, str]:
    ts_hint = project_root / 'artifacts' / 'purity_small_torchscript_path.txt'
    if ts_hint.exists():
        ts_path = Path(ts_hint.read_text(encoding='utf-8').strip()).resolve()
        if ts_path.exists():
            return ts_path, 'torchscript'

    ts_candidates = sorted((project_root / 'artifacts' / 'purity_notebook_export').glob('*_torchscript.pt'))
    if ts_candidates and use_torchscript:
        return ts_candidates[-1], 'torchscript'
    if use_torchscript:
        raise FileNotFoundError('TorchScript handle requested, but no TorchScript artifact was found.')

    eager_hint = project_root / 'artifacts' / 'purity_small_model_path.txt'
    if eager_hint.exists():
        eager_path = Path(eager_hint.read_text(encoding='utf-8').strip()).resolve()
        if eager_path.exists() and eager_path.suffix == '.pt' and not eager_path.name.endswith('_torchscript.pt'):
            return eager_path, 'purity_eager'

    state_candidates = sorted((project_root / 'artifacts' / 'purity_notebook_export').glob('*_state_dict.pt'))
    if state_candidates:
        return state_candidates[-1], 'purity_eager'

    if ts_candidates:
        return ts_candidates[-1], 'torchscript'

    raise FileNotFoundError('No exported PURITY model artifact found. Run training notebook first.')


input_sources = resolve_input_sources(
    DATA_DIR,
    use_all_files=USE_ALL_FILES,
    single_file=SINGLE_FILE,
    max_files=MAX_FILES,
)

model_path, model_handle_type = resolve_model_artifact(
    PROJECT_ROOT,
    use_torchscript=USE_TORCHSCRIPT_HANDLE,
)

print(f'Using {len(input_sources)} parquet input file(s):')
for src in input_sources:
    print(' -', src)
print('Model handle type:', model_handle_type)
print('Model path:', model_path)

input_sources, model_path, model_handle_type



Using 8 parquet input file(s):
 - /workspace/data/purity_inputs/all_ml_000.parquet
 - /workspace/data/purity_inputs/all_ml_001.parquet
 - /workspace/data/purity_inputs/all_ml_002.parquet
 - /workspace/data/purity_inputs/all_ml_003.parquet
 - /workspace/data/purity_inputs/all_ml_004.parquet
 - /workspace/data/purity_inputs/all_ml_005.parquet
 - /workspace/data/purity_inputs/all_ml_006.parquet
 - /workspace/data/purity_inputs/all_ml_007.parquet
Model handle type: torchscript
Model path: /workspace/artifacts/purity_small_export/purity_small_20260420_091344_20260420_091601_torchscript.pt


(['/workspace/data/purity_inputs/all_ml_000.parquet',
  '/workspace/data/purity_inputs/all_ml_001.parquet',
  '/workspace/data/purity_inputs/all_ml_002.parquet',
  '/workspace/data/purity_inputs/all_ml_003.parquet',
  '/workspace/data/purity_inputs/all_ml_004.parquet',
  '/workspace/data/purity_inputs/all_ml_005.parquet',
  '/workspace/data/purity_inputs/all_ml_006.parquet',
  '/workspace/data/purity_inputs/all_ml_007.parquet'],
 PosixPath('/workspace/artifacts/purity_small_export/purity_small_20260420_091344_20260420_091601_torchscript.pt'),
 'torchscript')

## Patch Config and Run

In [3]:
cfg = load_config()['inference']

# Guard knobs (inference should preserve row alignment by default)
GUARD_ROWS_ENABLED = True
GUARD_ROWS_TRAINING_ONLY = True  # Keep True so inference does not drop rows.
GUARD_REQUIRE_NONEMPTY_ATAR = True
GUARD_REQUIRE_NONEMPTY_TOTAL_HITS = True
GUARD_MAX_TOTAL_HITS = 300  # Set None to disable max-hit filtering.
GUARD_REQUIRE_FINITE_SCALARS = [
    'truth_theta',
    'truth_phi',
    'truth_positron_energy',
    'truth_pion_stop_x',
    'truth_pion_stop_y',
    'truth_pion_stop_z',
]


def patch_guard_knobs(section_cfg: dict) -> None:
    lm = dict(section_cfg.get('loader_manager') or {})
    lm_cfg = dict(lm.get('config') or {})

    def apply_guard_params(target_cfg: dict) -> dict:
        out = dict(target_cfg or {})
        out['guard_rows_enabled'] = bool(GUARD_ROWS_ENABLED)
        out['guard_rows_training_only'] = bool(GUARD_ROWS_TRAINING_ONLY)
        out['guard_require_nonempty_atar'] = bool(GUARD_REQUIRE_NONEMPTY_ATAR)
        out['guard_require_nonempty_total_hits'] = bool(GUARD_REQUIRE_NONEMPTY_TOTAL_HITS)
        out['guard_max_total_hits'] = GUARD_MAX_TOTAL_HITS
        out['guard_require_finite_scalars'] = list(GUARD_REQUIRE_FINITE_SCALARS)
        return out

    defaults = dict(lm_cfg.get('defaults') or {})
    defaults_cfg = apply_guard_params(dict(defaults.get('config') or {}))
    defaults['config'] = defaults_cfg
    lm_cfg['defaults'] = defaults

    loaders = dict(lm_cfg.get('loaders') or {})
    for lname, lspec in list(loaders.items()):
        lspec_dict = dict(lspec or {})
        lcfg = apply_guard_params(dict(lspec_dict.get('config') or {}))
        lspec_dict['config'] = lcfg
        loaders[lname] = lspec_dict
    lm_cfg['loaders'] = loaders

    lm_cfg = apply_guard_params(lm_cfg)

    lm['config'] = lm_cfg
    section_cfg['loader_manager'] = lm


cfg['model_handle_builder']['model_handle']['type'] = str(model_handle_type)
cfg['model_handle_builder']['model_handle']['config']['model_path'] = str(model_path)
cfg['inference']['loader_manager']['config']['input_sources_spec']['main_sources'] = list(input_sources)
cfg['inference']['loader_manager']['config']['input_sources_spec']['optional_sources_by_name'] = {}
cfg['inference']['loader_manager']['config']['input_sources_spec']['source_type'] = 'file'
patch_guard_knobs(cfg['inference'])

cfg['inference']['writer']['config']['output_dir'] = str(PROJECT_ROOT / 'artifacts' / 'purity_notebook_predictions')
cfg['inference']['writer']['config']['fallback_output_dir'] = str(PROJECT_ROOT / 'artifacts' / 'purity_notebook_predictions')
cfg['inference']['writer']['config']['write_timestamped'] = False

run = inference_pipeline.with_options(enable_cache=False)(pipeline_config=cfg)
out = load_step_output(run, 'run_inference')
out



Initiating a new run for the pipeline: inference_pipeline.
Caching is disabled by default for inference_pipeline.
Using user: default
Using stack: default
  deployer: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step build_model_handle has started.
Step build_model_handle has finished in 0.184s.
Step run_inference has started.
Step run_inference has finished in 1m49s.
Pipeline run has finished in 1m52s.


{'predictions_path': None,
 'predictions_paths': ['/workspace/artifacts/purity_notebook_predictions/all_ml_000_preds.parquet',
  '/workspace/artifacts/purity_notebook_predictions/all_ml_001_preds.parquet',
  '/workspace/artifacts/purity_notebook_predictions/all_ml_002_preds.parquet',
  '/workspace/artifacts/purity_notebook_predictions/all_ml_003_preds.parquet',
  '/workspace/artifacts/purity_notebook_predictions/all_ml_004_preds.parquet',
  '/workspace/artifacts/purity_notebook_predictions/all_ml_005_preds.parquet',
  '/workspace/artifacts/purity_notebook_predictions/all_ml_006_preds.parquet',
  '/workspace/artifacts/purity_notebook_predictions/all_ml_007_preds.parquet'],
 'timestamped_predictions_path': None,
 'timestamped_predictions_paths': []}

In [4]:
out = load_step_output(run, 'run_inference')
print(out)

pred_path_str = (
    out.get('predictions_path')
    or (out.get('predictions_paths') or [None])[0]
    or out.get('timestamped_predictions_path')
    or (out.get('timestamped_predictions_paths') or [None])[0]
)
if pred_path_str is None:
    raise RuntimeError(f"No prediction file path found in run_inference output: {out}")

pred_path = Path(pred_path_str)
tbl = pq.read_table(pred_path)
print(pred_path)
print(tbl.schema)
tbl.slice(0, 3).to_pydict()


{'predictions_path': None, 'predictions_paths': ['/workspace/artifacts/purity_notebook_predictions/all_ml_000_preds.parquet', '/workspace/artifacts/purity_notebook_predictions/all_ml_001_preds.parquet', '/workspace/artifacts/purity_notebook_predictions/all_ml_002_preds.parquet', '/workspace/artifacts/purity_notebook_predictions/all_ml_003_preds.parquet', '/workspace/artifacts/purity_notebook_predictions/all_ml_004_preds.parquet', '/workspace/artifacts/purity_notebook_predictions/all_ml_005_preds.parquet', '/workspace/artifacts/purity_notebook_predictions/all_ml_006_preds.parquet', '/workspace/artifacts/purity_notebook_predictions/all_ml_007_preds.parquet'], 'timestamped_predictions_path': None, 'timestamped_predictions_paths': []}
/workspace/artifacts/purity_notebook_predictions/all_ml_000_preds.parquet
event_id: int64
pred_purity_signal: list<element: float>
  child 0, element: float
pred_purity_logit: list<element: float>
  child 0, element: float
pred_purity_summary_accepted: list<e

{'event_id': [0, 1, 2],
 'pred_purity_signal': [[0.5260536670684814,
   0.5779711604118347,
   0.5256346464157104,
   0.5787976980209351,
   0.4897456169128418,
   0.3954487144947052,
   0.5792998671531677],
  [0.48937273025512695, 0.3963281810283661],
  [0.418367862701416,
   0.5252313613891602,
   0.5836020708084106,
   0.49258288741111755,
   0.5330544710159302,
   0.5826461911201477,
   0.4914805591106415]],
 'pred_purity_logit': [[0.10430917143821716,
   0.31445032358169556,
   0.10262855887413025,
   0.31783992052078247,
   -0.0410233736038208,
   -0.42446523904800415,
   0.31989991664886475],
  [-0.04251560568809509, -0.4207879900932312],
  [-0.3294770121574402,
   0.10101130604743958,
   0.3375779986381531,
   -0.029670536518096924,
   0.13241097331047058,
   0.333645761013031,
   -0.03408098220825195]],
 'pred_purity_summary_accepted': [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
  [0.0, 0.0],
  [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]],
 'pred_purity_summary_positron_energy': [[0.5639424